In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [ ]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

# The 'path' variable already holds the correct base directory:
# path = '/kaggle/input/imdb-dataset-of-50k-movie-reviews'
# So, we just need to join the filename directly to it.
dataset_path = os.path.join(path, 'IMDB Dataset.csv')

# Load the dataset into a pandas DataFrame
df = pd.read_csv(dataset_path)


In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:


train_df, test_df = train_test_split(
    df,
    test_size=0.2,          # 20% test
    random_state=42,
    stratify=df["sentiment"]  # Preserve class balance
)

In [ ]:
print(train_df["sentiment"].value_counts())

print(test_df["sentiment"].value_counts())

sentiment
positive    20000
negative    20000
Name: count, dtype: int64
sentiment
negative    5000
positive    5000
Name: count, dtype: int64


Load Data
      ↓
Train/Test Split
      ↓
Encode labels (y)
      ↓
Create tf.data.Dataset
      ↓
TextVectorization (X preprocessing)
      ↓
Batch
      ↓
Prefetch
      ↓
Train model

In [ ]:
train_df["sentiment"] = train_df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

test_df["sentiment"] = test_df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

In [ ]:
X_train = train_df["review"]
y_train = train_df["sentiment"]

In [ ]:
X_test=test_df["review"]
y_test=test_df["sentiment"]

create tf.data dataset

In [ ]:

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

Shuffle, batch, and prefetch:

In [ ]:
BATCH_SIZE = 32

train_ds = (
    train_ds
    .shuffle(buffer_size=len(train_df))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
for text_batch, label_batch in train_ds.take(1):
    print(text_batch.shape)
    print(label_batch.shape)
    print(text_batch[0].numpy()[:300])
    print(label_batch[0].numpy())

(32,)
(32,)
b"This is a stupid movie. Like a lot of these karate movies it is badly written, awkward, and sometimes just stupid. The action really isn't all there and the movie overall leaves much to be desired. Everyone here is talking jive, doing bad karate and doing a very bad job of acting. <br /><br />Watch "
0


Text vectorisation

In [ ]:
MAX_VOCAB = 10000 #keep the most frequent tokens 10000 Rare words become an OOV (Out-Of-Vocabulary) token
#create a layer
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int"
)
#learn the vocabulary
vectorizer.adapt(train_ds.map(lambda text, label: text))
#inspect the vocabulary

vocab = vectorizer.get_vocabulary()

print(vocab[:20])

['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('br'), np.str_('was'), np.str_('as'), np.str_('for'), np.str_('with'), np.str_('movie'), np.str_('but')]


In [ ]:
def vectorize_text(text, label):
    text = vectorizer(text)
    return text, label

In [ ]:
train_vectorized = train_ds.map(vectorize_text)

test_vectorized = test_ds.map(vectorize_text)

In [ ]:
train_vectorized

<_MapDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [ ]:

EMBEDDING_DIM = 128  #every word is represented by 128 learned features.
#The numbers while vectorising are just IDs. They have no semantic meaning.
#The Embedding layer learns a dense vector for every word.

#After the embedding layer, one review has shape:(250,128) That means:250 words each represented by a 128-dimensional vector

model = tf.keras.Sequential([   #the model starts with the embedding
    tf.keras.layers.Embedding(
        input_dim=MAX_VOCAB,
        output_dim=EMBEDDING_DIM
    ),
    tf.keras.layers.GlobalAveragePooling1D(),  ##it solves this by computing the average across all 250 word vectors. because A Dense layer cannot directly process this 2D sequence—
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

NameError: name 'history_v2' is not defined

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_vectorized,
    validation_data=test_vectorized,
    epochs=5
)



In [ ]:
print('Evaluating Model 1 on training data:')
loss_train, accuracy_train = model.evaluate(train_vectorized)
print(f'Training Loss: {loss_train:.4f}')
print(f'Training Accuracy: {accuracy_train:.4f}')

In [ ]:
print('\nEvaluating Model 1 on test data:')
loss_test, accuracy_test = model.evaluate(test_vectorized)
print(f'Test Loss: {loss_test:.4f}')
print(f'Test Accuracy: {accuracy_test:.4f}')

The difference is about 1.8%. This is a relatively small gap. While there's a slight drop in performance from training to test, it doesn't indicate severe overfitting. A healthy model will usually perform slightly better on the training data than on the test data.

However, it suggests there might be a minor degree of overfitting. The changes introduced in model_v2 (adding LSTM layers and Dropout) were specifically aimed at improving generalization and reducing overfitting, which should lead to a smaller gap between training and validation/test accuracy.

In [ ]:
import matplotlib.pyplot as plt

# Get training and validation loss from history
loss = history.history['loss']

epochs = range(1, len(loss) + 1)

# Plotting the loss
plt.plot(epochs, loss, '--', label='Training loss')

plt.title('Training')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
new_reviews = tf.constant([
    "This movie was amazing!",
    "This was boring and terrible.",
    "I was not excited but ok "
])

new_reviews_vectorized = vectorizer(new_reviews)

predictions = model.predict(new_reviews_vectorized)

print(predictions)

### Improving Model Performance

To enhance the model's validation accuracy, we can implement several strategies:

1.  **Integrate `TextVectorization`**: Include the `TextVectorization` layer directly into the `tf.keras.Sequential` model. This makes the model self-contained for deployment.
2.  **Use `LSTM` Layers**: Replace `GlobalAveragePooling1D` with `LSTM` (Long Short-Term Memory) layers. LSTMs are a type of Recurrent Neural Network (RNN) particularly effective at capturing long-range dependencies and contextual information in sequential data like text, which `GlobalAveragePooling1D` might miss.
3.  **Add `Dropout` Regularization**: Introduce `Dropout` layers after the `Embedding` and `LSTM` layers. Dropout randomly sets a fraction of input units to zero at each update during training, which helps prevent overfitting by making the model less reliant on specific neurons and forcing it to learn more robust features.

### Model 2

In [ ]:
# Define the new model with TextVectorization, Embedding, GlobalAveragePooling1D, and Dropout layers
model_v2 = tf.keras.Sequential([
    vectorizer, # Include the TextVectorization layer directly in the model
    tf.keras.layers.Embedding(
        input_dim=MAX_VOCAB,
        output_dim=EMBEDDING_DIM
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2), # Add dropout after pooling
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_v2.summary()

In [ ]:
# Compile the new model
model_v2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the new model
history_v2 = model_v2.fit(
    train_ds, # Use the original train_ds as TextVectorization is now inside the model
    validation_data=test_ds, # Use the original test_ds
    epochs=5
)

In [ ]:
import matplotlib.pyplot as plt

# Get training and validation loss from history_v2
loss_v2 = history_v2.history['loss']
val_loss_v2 = history_v2.history['val_loss']

# Get training and validation accuracy from history_v2
accuracy_v2 = history_v2.history['accuracy']
val_accuracy_v2 = history_v2.history['val_accuracy']

epochs_v2 = range(1, len(loss_v2) + 1)

plt.figure(figsize=(12, 5))

# Plotting the loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
plt.plot(epochs_v2, loss_v2, 'bo-', label='Training loss')
plt.plot(epochs_v2, val_loss_v2, 'ro-', label='Validation loss')
plt.title('Training and Validation Loss (Model V2)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plotting the accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
plt.plot(epochs_v2, accuracy_v2, 'bo-', label='Training accuracy')
plt.plot(epochs_v2, val_accuracy_v2, 'ro-', label='Validation accuracy')
plt.title('Training and Validation Accuracy (Model V2)')
plt.xlabel('Epochs')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
print('Evaluating Model 1 on training data:')
loss_train, accuracy_train = model_v2.evaluate(train_ds)
print(f'Training Loss: {loss_train:.4f}')
print(f'Training Accuracy: {accuracy_train:.4f}')

In [ ]:
print('\nEvaluating Model 2 on test data:')
loss_test_v2, accuracy_test_v2 = model_v2.evaluate(test_ds) # Correctly using test_ds for model_v2
print(f'Test Loss (Model 2): {loss_test_v2:.4f}')
print(f'Test Accuracy (Model 2): {accuracy_test_v2:.4f}')

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Get predictions for model_v2 on test_ds
y_pred_probabilities_v2 = model_v2.predict(test_ds)
y_pred_v2 = (y_pred_probabilities_v2 > 0.5).astype(int)

# Get true labels from test_ds
y_true_v2 = np.concatenate([y for x, y in test_ds], axis=0)

In [ ]:
# Compute the confusion matrix
cm_v2 = confusion_matrix(y_true_v2, y_pred_v2)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_v2, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Model V2 on Test Data')
plt.show()

### Precision and Recall

Precision measures the accuracy of positive predictions (out of all predicted positives, how many were actually positive). Recall measures the completeness of positive predictions (out of all actual positives, how many were correctly identified).

We can use `sklearn.metrics` to calculate these, along with the F1-score, which is the harmonic mean of precision and recall.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Calculate Precision, Recall, and F1-score for Model V2
precision_v2 = precision_score(y_true_v2, y_pred_v2)
recall_v2 = recall_score(y_true_v2, y_pred_v2)
f1_v2 = f1_score(y_true_v2, y_pred_v2)

print(f"Precision (Model V2): {precision_v2:.4f}")
print(f"Recall (Model V2): {recall_v2:.4f}")
print(f"F1-Score (Model V2): {f1_v2:.4f}")

### LSTM

In [ ]:

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=MAX_VOCAB,
        output_dim=EMBEDDING_DIM
    ),

    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(64)
    ),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_lstm = lstm_model.fit(
    train_vectorized,
    validation_data=test_vectorized,
    epochs=5
)

Epoch 1/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1414s 1s/step - accuracy: 0.7653 - loss: 0.5018 - val_accuracy: 0.6284 - val_loss: 0.7042
Epoch 2/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1410s 1s/step - accuracy: 0.7897 - loss: 0.4789 - val_accuracy: 0.5310 - val_loss: 0.6832
Epoch 3/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1438s 1s/step - accuracy: 0.8235 - loss: 0.4140 - val_accuracy: 0.8408 - val_loss: 0.3720
Epoch 4/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1413s 1s/step - accuracy: 0.7365 - loss: 0.4951 - val_accuracy: 0.8783 - val_loss: 0.3232
Epoch 5/5
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1494s 1s/step - accuracy: 0.9000 - loss: 0.2925 - val_accuracy: 0.8925 - val_loss: 0.2790


In [ ]:
print('Evaluating Model 1 on training data:')
loss_train, accuracy_train =lstm_model.evaluate(train_ds)
print(f'Training Loss: {loss_train:.4f}')
print(f'Training Accuracy: {accuracy_train:.4f}')

Evaluating Model 1 on training data:


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("sequential_3_1/Cast:0", shape=(None,), dtype=float32) with name 'keras_tensor_6' and path ''. Expected shape (None, None), but input has incompatible shape (None,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None,), dtype=string)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>